# 🥔 আলুর রোগ নির্ণয় — MobileNetV3-Large (Kaggle → server)

**Dataset:** PLD (খেতের ছবি) + PlantVillage-এর আলু + আপনার `not_leaf` ছবি

**এই notebook কী করে**
1. PLD (`Early_Blight`) আর PlantVillage (`Potato___Early_blight`) এর folder app-এর label-এ মেলায়: `early_blight`, `late_blight`, `healthy`
2. `not_leaf` class বানায়: আপনার পাতাহীন ছবি + PlantVillage-এর টমেটো ও মরিচের পাতা ("এটি আলুর পাতা নয়")
3. PLD-র নিজের Training/Validation/Testing ভাগ যেমন আছে রাখে; বাকি ছবি stratified ভাগ
4. MobileNetV3-Large train (label smoothing + MixUp)। TFLite ~৬ MB, EfficientNetV2-B2-র (~১৭ MB) প্রায় এক-তৃতীয়াংশ; EfficientNet-এর অংশ comment করা আছে
5. Test-এ মূল্যায়ন — **PLD test আলাদা করে দেখায়** (খেতের ছবি, আসল মাপ)
6. **`potato_server_bundle.zip`** বানায় → খুললেই `potato/model.tflite`, `labels.txt`, `meta.json`

> Kaggle Notebook → Settings → **Accelerator: GPU (T4/P100)** আর **Internet: On** করে নিন।
>
> লম্বা training-এ **Save Version → Save & Run All (Commit)** দিন — বিদ্যুৎ/internet চলে গেলেও Kaggle-এর server-এ চলতে থাকবে, ফলাফল Output-এ জমা থাকবে।
>
> **আগের run-এর MobileNet আবার train করতে হবে না:** আগের version-এর Output-এ `potato_best_mobilenetv3large.keras` আছে। এই notebook-এ **Add Input → Your Work → এই notebook-এর আগের version** যোগ করুন — `RESUME = True` সেটা নিজেই খুঁজে নিয়ে training ছাড়া ~১০ মিনিটে মূল্যায়ন ও TFLite export করবে। না পেলে নিজে থেকেই নতুন করে train করবে।

## ১. Setup ও Config

In [ ]:
import os, random, json, shutil, zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import matplotlib.pyplot as plt
from PIL import Image
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.utils.class_weight import compute_class_weight

print("TensorFlow:", tf.__version__, "| Keras:", keras.__version__)
print("GPU:", tf.config.list_physical_devices("GPU"))

SEED = 42
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)

# ---------------- আপনার dataset path ----------------
# PLD: ভেতরে Training / Validation / Testing, প্রতিটায় Early_Blight / Healthy / Late_Blight
PLD_ROOT = "/kaggle/input/datasets/rizwan123456789/potato-disease-leaf-datasetpld/PLD_3_Classes_256"

# PlantVillage: Potato___Early_blight / Potato___Late_blight / Potato___healthy নেবে
PV_BASE = "/kaggle/input/datasets/arjuntejaswi/plant-village/PlantVillage"
USE_PLANTVILLAGE = True

# "পাতা নয়" ছবি (মাটি, হাত, আকাশ, অন্য জিনিস) — ভেতরের সব ছবি নেবে
NOT_LEAF_DIR = "/kaggle/input/datasets/jihadulislam1/without-leaves-full"

# PlantVillage-এর টমেটো ও মরিচের পাতাও not_leaf-এ যোগ হবে, যাতে অন্য গাছের পাতা দিলে
# model "আলুর পাতা নয়" বলে। প্রতি folder থেকে সর্বোচ্চ এতগুলো ছবি (১২টা folder)।
OTHER_CROPS_AS_NOT_LEAF = True
OTHER_CROP_PER_FOLDER = 120

# যে class বাদ দিতে চান (app-এর label নামে), যেমন ["healthy"]
DROP_CLASSES = []

IMG = 256                    # PLD আর PlantVillage দুটোই 256px
BATCH = 32
EPOCHS_HEAD = 8             # শুধু নতুন layer train
EPOCHS_FT = 25               # fine-tuning (early stopping আছে)
# EfficientNet বন্ধ: test-এ MobileNet সমান ভালো (99.29% বনাম 98.58%), TFLite ~৬ MB বনাম ~১৭ MB।
# আবার তুলনা করতে চাইলে নিচের লাইনের comment তুলে দিন (SPECS-এও)।
# CANDIDATES = ["efficientnetv2b2", "mobilenetv3large"]
CANDIDATES = ["mobilenetv3large"]
LABEL_SMOOTHING = 0.1        # overfitting কমায়
MIXUP_ALPHA = 0.2            # 0 দিলে MixUp বন্ধ

# ---- আগের save করা model থেকে চালু (training বাদ) ----
# আগের run-এর output (potato_best_mobilenetv3large.keras আর outputs/split_*.csv) input-এ
# যোগ করুন: Add Input → Your Work → এই notebook। True হলে training বাদ, সরাসরি মূল্যায়ন ও export;
# checkpoint না পেলে নিজে থেকেই নতুন করে train হবে।
RESUME = True
# checkpoint-এর folder; খালি রাখলে পুরো /kaggle/input-এ খুঁজবে
RESUME_DIR = ""

RESUME_MODEL = ""
if RESUME:
    search = RESUME_DIR if os.path.isdir(RESUME_DIR) else "/kaggle/input"
    # শুধু CANDIDATES-এর model-এর checkpoint (potato_best_mobilenetv3large.keras), আলুরটা আগে
    hits = sorted((p for p in Path(search).rglob("*.keras")
                   if any(c in p.name.lower() for c in CANDIDATES)),
                  key=lambda p: ("potato" not in p.name.lower(), str(p)))
    if hits:
        RESUME_MODEL = str(hits[0])
        RESUME_DIR = str(hits[0].parent.parent if hits[0].parent.name == "outputs" else hits[0].parent)
        print("⏩ আগের model থেকে চালু:", RESUME_MODEL)
    else:
        print(f"⚠️ {CANDIDATES}-এর .keras পাওয়া যায়নি — নতুন করে train হবে")
        RESUME = False

WORK = Path("/kaggle/working")
OUT = WORK / "outputs"          # সব ফলাফল এখানে জমা হয়, শেষে এক zip-এ
OUT.mkdir(exist_ok=True)

def savefig(name):
    plt.savefig(OUT / name, dpi=150, bbox_inches="tight")
EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

## ২. Dataset খুঁজে বের করা
PLD-তে `Early_Blight`, PlantVillage-এ `Potato___Early_blight` — নাম ছোট করে মিলিয়ে একই label দেওয়া হয়।

In [ ]:
import re

def norm(name):
    return re.sub(r"_+", "_", name.strip().lower().replace(" ", "_"))

# app: lib/data/library/potato_disease_info.dart, server: models/potato/labels.txt
POTATO_LABELS = {"early_blight", "late_blight", "healthy"}

def label_for(folder_name):
    n = norm(folder_name)
    if n.startswith("potato_"):
        n = n[len("potato_"):]
    return n if n in POTATO_LABELS else None

def pv_label(folder_name):
    # PlantVillage-এ শুধু "Potato..." folder আলু
    return label_for(folder_name) if norm(folder_name).startswith("potato") else None

def list_images(folder):
    return [p for p in Path(folder).rglob("*") if p.suffix.lower() in EXTS]

def find_dir(base, want):
    # base-এর ভেতরে 'want' নামের folder (ছোট-বড় হাতের পার্থক্য বাদ), যার ভেতরে আলুর class আছে
    for root, dirs, _ in os.walk(base):
        for d in dirs:
            p = Path(root) / d
            if d.lower() == want.lower() and any(label_for(c.name) for c in p.iterdir() if c.is_dir()):
                return p
    return None

if not os.path.exists(PLD_ROOT):
    print("⚠️ PLD_ROOT পাওয়া যায়নি, /kaggle/input-এ খুঁজছি...")
    t = find_dir("/kaggle/input", "Training")
    assert t, "PLD-র Training folder পাওয়া যায়নি — PLD_ROOT দেখুন"
    PLD_ROOT = str(t.parent)
PLD_SPLITS = {}
for split, name in [("train", "Training"), ("val", "Validation"), ("test", "Testing")]:
    d = find_dir(PLD_ROOT, name)
    assert d, f"PLD-র {name} folder নেই: {PLD_ROOT}"
    PLD_SPLITS[split] = d
print("PLD:", PLD_ROOT)
for split, d in PLD_SPLITS.items():
    print(f"  {split:5s}", {c.name: label_for(c.name) for c in sorted(d.iterdir()) if c.is_dir()})

PV_ROOT = None
if USE_PLANTVILLAGE:
    search_base = PV_BASE if os.path.exists(PV_BASE) else "/kaggle/input"
    if search_base != PV_BASE:
        print("⚠️ PV_BASE পাওয়া যায়নি, পুরো /kaggle/input-এ খুঁজছি...")
    best_n = 0
    for root, dirs, _ in os.walk(search_base):
        n = sum(pv_label(d) is not None for d in dirs)
        if n > best_n:
            PV_ROOT, best_n = root, n
    assert PV_ROOT and best_n == 3, "PlantVillage-এর আলুর ৩টা folder পাওয়া যায়নি — PV_BASE দেখুন"
    print("\nPlantVillage:", PV_ROOT)
    for d in sorted(os.listdir(PV_ROOT)):
        other = OTHER_CROPS_AS_NOT_LEAF and norm(d).startswith(("tomato", "pepper"))
        print(f"  {d:45s} -> {pv_label(d) or ('not_leaf' if other else '—')}")

## ৩. ছবি সংগ্রহ

In [ ]:
records = []   # (path, label, source, PLD-র ভাগ)
for split, d in PLD_SPLITS.items():
    for c in sorted(d.iterdir()):
        label = label_for(c.name) if c.is_dir() else None
        if label and label not in DROP_CLASSES:
            records += [(str(p), label, "pld", split) for p in list_images(c)]

rng = random.Random(SEED)
if PV_ROOT:
    for d in sorted(Path(PV_ROOT).iterdir()):
        if not d.is_dir():
            continue
        label = pv_label(d.name)
        if label:
            if label not in DROP_CLASSES:
                records += [(str(p), label, "plantvillage", None) for p in list_images(d)]
        elif OTHER_CROPS_AS_NOT_LEAF and norm(d.name).startswith(("tomato", "pepper")):
            imgs = list_images(d)
            rng.shuffle(imgs)
            records += [(str(p), "not_leaf", "other_crop", None) for p in imgs[:OTHER_CROP_PER_FOLDER]]

assert os.path.exists(NOT_LEAF_DIR), f"NOT_LEAF_DIR পাওয়া যায়নি: {NOT_LEAF_DIR}"
records += [(str(p), "not_leaf", "no_leaf", None) for p in list_images(NOT_LEAF_DIR)]

df = pd.DataFrame(records, columns=["path", "label", "source", "split"])
print("মোট ছবি:", len(df))

# Dataset পরিষ্কার, তাই নষ্ট/duplicate যাচাই বাদ — সরাসরি training-এ
CLASSES = sorted(df["label"].unique())
CLASS_TO_IDX = {c: i for i, c in enumerate(CLASSES)}
df["y"] = df["label"].map(CLASS_TO_IDX)
print("\nClasses:", CLASSES)
print(pd.crosstab(df["label"], df["source"], margins=True))

In [ ]:
# প্রতি class-এর কয়েকটা ছবি দেখে নিন (ঠিক folder-এ ঠিক রোগ আছে কি না যাচাই)
fig, axes = plt.subplots(len(CLASSES), 4, figsize=(10, 2.4 * len(CLASSES)))
for r, c in enumerate(CLASSES):
    sample = df[df.label == c].sample(4, random_state=1)["path"].tolist()
    for k, p in enumerate(sample):
        axes[r, k].imshow(Image.open(p).convert("RGB"))
        axes[r, k].axis("off")
        if k == 0:
            axes[r, k].set_title(c, fontsize=9, loc="left")
plt.tight_layout(); savefig("01_class_samples.png"); plt.show()

## ৪. Train / Val / Test ভাগ
PLD-র নিজের Training/Validation/Testing যেমন আছে তেমনই থাকে। PlantVillage আর not_leaf ছবি 70/15/15 stratified ভাগ। Augmentation শুধু train-এ।

In [ ]:
# RESUME-এ আগের run-এর split_*.csv পাওয়া গেলে হুবহু সেই ভাগ, নইলে একই seed দিয়ে আবার ভাগ
def prev_split(n):
    return next(Path(RESUME_DIR).rglob(f"split_{n}.csv"), None) if RESUME else None

prev = {n: prev_split(n) for n in ("train", "val", "test")}
if all(prev.values()):
    train_df, val_df, test_df = (pd.read_csv(prev[n]) for n in ("train", "val", "test"))
    for _p in (train_df, val_df, test_df):
        _p["y"] = _p["label"].map(CLASS_TO_IDX)
    assert all(os.path.exists(p) for p in test_df["path"][:20]), "split_*.csv-এর path এই session-এ নেই"
    print("আগের run-এর split_*.csv থেকে একই ভাগ নেওয়া হলো")
else:
    pld = df[df.source == "pld"]
    rest = df[df.source != "pld"]
    strata = rest["label"] + "|" + rest["source"]
    r_train, r_tmp = train_test_split(rest, test_size=0.30, stratify=strata, random_state=SEED)
    r_val, r_test = train_test_split(
        r_tmp, test_size=0.50, stratify=strata.loc[r_tmp.index], random_state=SEED)

    train_df = pd.concat([pld[pld.split == "train"], r_train]).reset_index(drop=True)
    val_df = pd.concat([pld[pld.split == "val"], r_val]).reset_index(drop=True)
    test_df = pd.concat([pld[pld.split == "test"], r_test]).reset_index(drop=True)

for name, part in [("train", train_df), ("val", val_df), ("test", test_df)]:
    print(name, len(part))
counts = pd.DataFrame({
    "train": train_df.label.value_counts(),
    "val": val_df.label.value_counts(),
    "test": test_df.label.value_counts(),
})
print(counts)
print("\nTest-এ কোন উৎসের কত ছবি:")
print(pd.crosstab(test_df["label"], test_df["source"]))

counts.to_csv(OUT / "class_counts.csv")
for _n, _p in [("train", train_df), ("val", val_df), ("test", test_df)]:
    _p.to_csv(OUT / f"split_{_n}.csv", index=False)

## ৫. tf.data pipeline + augmentation + MixUp
Label এখন one-hot (label smoothing আর MixUp-এর জন্য দরকার)।

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE
N_CLASSES = len(CLASSES)

def load_image(path, label):
    raw = tf.io.read_file(path)
    img = tf.io.decode_image(raw, channels=3, expand_animations=False)
    img.set_shape([None, None, 3])
    img = tf.image.resize(img, (IMG, IMG))          # 0..255 float
    return tf.cast(img, tf.float32), label

# মাঠের অবস্থা অনুকরণ: ঘোরানো, জুম, আলো-ছায়া, কনট্রাস্ট
augment = keras.Sequential([
    layers.RandomFlip("horizontal_and_vertical"),
    layers.RandomRotation(0.2),
    layers.RandomZoom((-0.15, 0.1)),   # বেশি জুমে দাগ কেটে যায়
    layers.RandomTranslation(0.1, 0.1),
    layers.RandomBrightness(0.3, value_range=(0, 255)),
    layers.RandomContrast(0.3),
], name="augment")

def sample_beta(alpha, n):
    g1 = tf.random.gamma([n], alpha)
    g2 = tf.random.gamma([n], alpha)
    return g1 / (g1 + g2)

# MixUp: দুটো ছবি হালকা মিশিয়ে label-ও মেশায়, model অতি-আত্মবিশ্বাসী হয় না
def mixup(x, y):
    n = tf.shape(x)[0]
    lam = sample_beta(MIXUP_ALPHA, n)
    idx = tf.random.shuffle(tf.range(n))
    lx = tf.reshape(lam, [-1, 1, 1, 1])
    ly = tf.reshape(lam, [-1, 1])
    return x * lx + tf.gather(x, idx) * (1 - lx), y * ly + tf.gather(y, idx) * (1 - ly)

def make_ds(frame, training):
    ds = tf.data.Dataset.from_tensor_slices(
        (frame["path"].values, frame["y"].values.astype("int32")))
    if training:
        ds = ds.shuffle(len(frame), seed=SEED, reshuffle_each_iteration=True)
    ds = ds.map(load_image, num_parallel_calls=AUTOTUNE)
    ds = ds.map(lambda x, y: (x, tf.one_hot(y, N_CLASSES)), num_parallel_calls=AUTOTUNE)
    ds = ds.batch(BATCH)
    if training:
        ds = ds.map(lambda x, y: (tf.clip_by_value(augment(x, training=True), 0., 255.), y),
                    num_parallel_calls=AUTOTUNE)
        if MIXUP_ALPHA > 0:
            ds = ds.map(mixup, num_parallel_calls=AUTOTUNE)
    return ds.prefetch(AUTOTUNE)

train_ds = make_ds(train_df, True)
val_ds = make_ds(val_df, False)
test_ds = make_ds(test_df, False)

# Class imbalance ঠিক করতে weight (one-hot label-এও কাজ করে)
cw = compute_class_weight("balanced", classes=np.arange(N_CLASSES), y=train_df["y"].values)
CLASS_WEIGHT = {i: float(w) for i, w in enumerate(cw)}
print("Class weights:", {CLASSES[i]: round(w, 2) for i, w in CLASS_WEIGHT.items()})

# Augmentation + MixUp কেমন দেখায়
x, y = next(iter(train_ds))
plt.figure(figsize=(10, 3))
for i in range(5):
    plt.subplot(1, 5, i + 1)
    top = int(tf.argmax(y[i]))
    plt.imshow(x[i].numpy().astype("uint8"))
    plt.title(f"{CLASSES[top]} ({float(y[i][top]):.0%})", fontsize=7); plt.axis("off")
savefig("02_augmentation_preview.png"); plt.show()

## ৬. Model
`include_preprocessing=True`, তাই **ইনপুট 0–255 float** (app-এ `/255` করা যাবে না)।

In [ ]:
SPECS = {
    # "efficientnetv2b2": dict(ctor=keras.applications.EfficientNetV2B2, unfreeze=200, lr_ft=5e-5),
    "mobilenetv3large": dict(ctor=keras.applications.MobileNetV3Large, unfreeze=130, lr_ft=5e-5),
}

def build_model(name):
    base = SPECS[name]["ctor"](
        input_shape=(IMG, IMG, 3), include_top=False, weights="imagenet",
        include_preprocessing=True)
    base.trainable = False
    inp = keras.Input((IMG, IMG, 3), name="image")   # 0..255 float32
    x = base(inp, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.3)(x)
    out = layers.Dense(N_CLASSES, activation="softmax", name="probs")(x)
    return keras.Model(inp, out, name=f"potato_{name}"), base

def compile_model(m, lr):
    m.compile(
        optimizer=keras.optimizers.Adam(lr),
        loss=keras.losses.CategoricalCrossentropy(label_smoothing=LABEL_SMOOTHING),
        metrics=[keras.metrics.CategoricalAccuracy(name="accuracy")],
    )

def train_candidate(name):
    spec = SPECS[name]
    m, base = build_model(name)

    # ধাপ ১: শুধু নতুন head
    compile_model(m, 1e-3)
    h1 = m.fit(train_ds, validation_data=val_ds, epochs=EPOCHS_HEAD,
               class_weight=CLASS_WEIGHT, callbacks=[
                   keras.callbacks.EarlyStopping(monitor="val_accuracy", patience=4,
                                                 restore_best_weights=True),
                   keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5,
                                                     patience=2, verbose=1),
               ])

    # ধাপ ২: fine-tuning, BatchNorm frozen
    base.trainable = True
    for layer in base.layers[:-spec["unfreeze"]]:
        layer.trainable = False
    for layer in base.layers:
        if isinstance(layer, layers.BatchNormalization):
            layer.trainable = False
    compile_model(m, spec["lr_ft"])
    ckpt = str(WORK / f"potato_best_{name}.keras")
    h2 = m.fit(train_ds, validation_data=val_ds, epochs=EPOCHS_FT,
               class_weight=CLASS_WEIGHT, callbacks=[
                   keras.callbacks.ModelCheckpoint(ckpt, monitor="val_accuracy",
                                                   save_best_only=True, verbose=1),
                   keras.callbacks.EarlyStopping(monitor="val_accuracy", patience=5,
                                                 restore_best_weights=True),
                   keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5,
                                                     patience=3, verbose=1),
               ])
    return m, h1, h2, ckpt

# TTA: আসল + বাম-ডান উল্টানো + উপর-নিচ উল্টানো, তিনটার গড়
def predict_tta(m, frame):
    ds = make_ds(frame, False)
    p = m.predict(ds, verbose=0)
    p += m.predict(ds.map(lambda x, y: (tf.image.flip_left_right(x), y)), verbose=0)
    p += m.predict(ds.map(lambda x, y: (tf.image.flip_up_down(x), y)), verbose=0)
    return p / 3

## ৭. Training
GPU-তে MobileNet ~২০–৩০ মিনিট। `RESUME`-এ আগের checkpoint পেলে training বাদ।

In [ ]:
def load_saved(path):
    # আগের run-এর ModelCheckpoint (.keras); না খুললে একই model বানিয়ে শুধু weights বসানো
    stem = Path(path).stem.lower()
    try:
        m = keras.models.load_model(path, compile=False)
    except Exception as e:
        print("load_model হয়নি, weights আলাদা করে বসাচ্ছি:", e)
        m, _ = build_model(next(k for k in SPECS if k in stem))
        m.load_weights(path)
    name = next((k for k in SPECS if k in f"{stem} {m.name}".lower()), stem)
    assert m.output_shape[-1] == N_CLASSES, \
        f"model-এ {m.output_shape[-1]}টা class, data-তে {N_CLASSES}টা — dataset/DROP_CLASSES আগের মতো রাখুন"
    return name, m

results = {}
y_val = val_df["y"].values

def evaluate_candidate(name, m, h1_, h2_, ckpt):
    val_plain = float((m.predict(val_ds, verbose=0).argmax(1) == y_val).mean())
    val_tta = float((predict_tta(m, val_df).argmax(1) == y_val).mean())
    results[name] = dict(model=m, h1=h1_, h2=h2_, ckpt=ckpt,
                         val_plain=val_plain, val_tta=val_tta)
    print(f"{name}: val {val_plain*100:.2f}% | val+TTA {val_tta*100:.2f}%")

if RESUME:
    name, m = load_saved(RESUME_MODEL)
    print("⏩ training বাদ — আগের model:", name)
    evaluate_candidate(name, m, None, None, RESUME_MODEL)
else:
    for name in CANDIDATES:
        print("\n" + "=" * 25, name, "=" * 25)
        evaluate_candidate(name, *train_candidate(name))

## ৮. ফলাফল (একাধিক model থাকলে তুলনা ও বাছাই)
বাছাই **val** দিয়ে। Test শুধু দেখানোর জন্য, তাই test সংখ্যা সৎ থাকে।

In [ ]:
y_test = test_df["y"].values
rows = []
for name, r in results.items():
    t_plain = float((r["model"].predict(test_ds, verbose=0).argmax(1) == y_test).mean())
    t_tta = float((predict_tta(r["model"], test_df).argmax(1) == y_test).mean())
    r["test_plain"], r["test_tta"] = t_plain, t_tta
    rows.append({"model": name, "params_M": round(r["model"].count_params() / 1e6, 2),
                 "val %": round(r["val_plain"] * 100, 2), "val+TTA %": round(r["val_tta"] * 100, 2),
                 "test %": round(t_plain * 100, 2), "test+TTA %": round(t_tta * 100, 2)})
comparison = pd.DataFrame(rows)
comparison.to_csv(OUT / "model_comparison.csv", index=False)
print(comparison.to_string(index=False))

BEST_NAME = max(results, key=lambda n: results[n]["val_plain"])
best = results[BEST_NAME]
model, h1, h2, BEST = best["model"], best["h1"], best["h2"], best["ckpt"]
# server প্রতি ছবি একবারই চালায়, তাই TTA শুধু তুলনার জন্য মাপা, export-এ বন্ধ
USE_TTA_FINAL = False
print(f"\nসেরা model: {BEST_NAME} | TTA: {'চালু' if USE_TTA_FINAL else 'বন্ধ'}")

In [ ]:
if h1 is None:
    print("আগের save করা model — training history নেই, তাই chart বাদ")
else:
    acc = h1.history["accuracy"] + h2.history["accuracy"]
    val_acc = h1.history["val_accuracy"] + h2.history["val_accuracy"]
    loss = h1.history["loss"] + h2.history["loss"]
    val_loss = h1.history["val_loss"] + h2.history["val_loss"]
    split = len(h1.history["accuracy"])

    pd.DataFrame({"accuracy": acc, "val_accuracy": val_acc, "loss": loss, "val_loss": val_loss}).to_csv(
        OUT / "training_history.csv", index_label="epoch")

    fig, ax = plt.subplots(1, 2, figsize=(12, 4))
    ax[0].plot(acc, label="train"); ax[0].plot(val_acc, label="val")
    ax[0].axvline(split - 1, ls="--", c="gray"); ax[0].set_title(f"Accuracy ({BEST_NAME})"); ax[0].legend()
    ax[1].plot(loss, label="train"); ax[1].plot(val_loss, label="val")
    ax[1].axvline(split - 1, ls="--", c="gray"); ax[1].set_title("Loss"); ax[1].legend()
    savefig("03_training_curves.png"); plt.show()

## ৯. Test set-এ সৎ মূল্যায়ন
Test ছবি training-এ কখনো দেখেনি। **এই সংখ্যাটাই আসল।**

In [ ]:
y_true = test_df["y"].values
probs_plain = model.predict(test_ds, verbose=0)
probs = predict_tta(model, test_df) if USE_TTA_FINAL else probs_plain
print("Model:", BEST_NAME, "| TTA:", USE_TTA_FINAL)
y_pred = probs.argmax(1)
conf = probs.max(1)

print("Test accuracy:", round((y_pred == y_true).mean() * 100, 2), "%\n")
report_txt = classification_report(y_true, y_pred, target_names=CLASSES, digits=3)
print(report_txt)
TEST_ACC = float((y_pred == y_true).mean())

# উৎস অনুযায়ী accuracy — PLD খেতের ছবি, তাই ওটাই আসল মাপ
src = test_df["source"].values
SOURCE_ACC = {}
for s in ["pld", "plantvillage", "other_crop", "no_leaf"]:
    m = src == s
    if m.any():
        SOURCE_ACC[s] = round(float((y_pred[m] == y_true[m]).mean()), 4)
        print(f"  {s:13s} {SOURCE_ACC[s]*100:6.2f}%  ({m.sum()}টা ছবি)")
report_txt += chr(10) + "Source accuracy: " + json.dumps(SOURCE_ACC)
(OUT / "classification_report.txt").write_text(
    f"Test accuracy: {TEST_ACC*100:.2f}%" + chr(10) * 2 + report_txt, encoding="utf-8")
json.dump(classification_report(y_true, y_pred, target_names=CLASSES, output_dict=True),
          open(OUT / "classification_report.json", "w"), indent=2)
pd.DataFrame({"path": test_df["path"].values,
              "actual": [CLASSES[i] for i in y_true],
              "predicted": [CLASSES[i] for i in y_pred],
              "confidence": conf}).to_csv(OUT / "test_predictions.csv", index=False)

cm = confusion_matrix(y_true, y_pred)
pd.DataFrame(cm, index=CLASSES, columns=CLASSES).to_csv(OUT / "confusion_matrix.csv")
fig, ax = plt.subplots(figsize=(10, 9))
ax.imshow(cm, cmap="Greens")
ax.set_xticks(range(len(CLASSES))); ax.set_yticks(range(len(CLASSES)))
ax.set_xticklabels(CLASSES, rotation=45, ha="right"); ax.set_yticklabels(CLASSES)
for i in range(len(CLASSES)):
    for j in range(len(CLASSES)):
        ax.text(j, i, cm[i, j], ha="center", va="center",
                color="white" if cm[i, j] > cm.max() / 2 else "black")
ax.set_xlabel("Predicted"); ax.set_ylabel("Actual"); plt.tight_layout()
savefig("04_confusion_matrix.png"); plt.show()

In [ ]:
# Confidence threshold ঠিক করা: threshold-এর ওপরে কতগুলো ছবিতে মত দেয় আর সেগুলো কত % ঠিক
thr_rows = []
print(f"{'threshold':>10} {'মত দেয় %':>10} {'সঠিক %':>10}")
for t in [0.4, 0.5, 0.6, 0.7, 0.8, 0.9]:
    m = conf >= t
    covered = m.mean() * 100
    correct = (y_pred[m] == y_true[m]).mean() * 100 if m.any() else float("nan")
    print(f"{t:>10} {covered:>10.1f} {correct:>10.1f}")
    thr_rows.append({"threshold": t, "covered_percent": round(covered, 1), "correct_percent": round(correct, 1)})
pd.DataFrame(thr_rows).to_csv(OUT / "confidence_thresholds.csv", index=False)

# server-এর threshold val দিয়ে ঠিক করা (test দিয়ে নয়, যাতে test সৎ থাকে):
# ০.৬ থেকে শুরু করে সবচেয়ে ছোট t যেখানে t-এর ওপরের ছবিগুলোর ≥৯৮% সঠিক।
# ০.৬-এর নিচে নয়: খেতের ছবি dataset-এর ছবির চেয়ে কঠিন, val-এ নিচু t ভালো দেখালেও মাঠে ঝুঁকি
val_probs = model.predict(val_ds, verbose=0)
v_pred, v_conf = val_probs.argmax(1), val_probs.max(1)
CONFIDENT_T = 0.6
for t in np.arange(0.60, 0.91, 0.05):
    mask = v_conf >= t
    if mask.any() and (v_pred[mask] == y_val[mask]).mean() >= 0.98:
        CONFIDENT_T = round(float(t), 2)
        break
MIN_T = round(max(0.40, CONFIDENT_T - 0.2), 2)
print(f"\nServer threshold → নিশ্চিত: ≥{CONFIDENT_T} | সম্ভাব্য: ≥{MIN_T}")

In [ ]:
# মাঠের কঠিন অবস্থার পরীক্ষা: অন্ধকার/ঝাপসা/noise যোগ করে test ছবি আবার চালানো
def degrade(x, y):
    x = tf.image.random_brightness(x, 50.0)
    x = tf.image.random_contrast(x, 0.6, 1.4)
    x = x + tf.random.normal(tf.shape(x), stddev=12.0)
    return tf.clip_by_value(x, 0., 255.), y

hard_ds = make_ds(test_df, False).map(degrade)
hard_probs = model.predict(hard_ds, verbose=0)
hard_acc = (hard_probs.argmax(1) == y_true).mean() * 100
print(f"পরিষ্কার test: {(y_pred == y_true).mean()*100:.1f}%  |  খারাপ অবস্থার test: {hard_acc:.1f}%")
HARD_ACC = float(hard_acc) / 100

## ১০. কোথায় ভুল করছে দেখুন

In [ ]:
wrong = np.where(y_pred != y_true)[0]
pd.DataFrame({"path": test_df.iloc[wrong]["path"].values,
              "actual": [CLASSES[i] for i in y_true[wrong]],
              "predicted": [CLASSES[i] for i in y_pred[wrong]],
              "confidence": conf[wrong]}).to_csv(OUT / "wrong_predictions.csv", index=False)
print("ভুল:", len(wrong), "/", len(y_true))
show = wrong[:12]
if len(show):
    cols = 4
    rows = int(np.ceil(len(show) / cols))
    plt.figure(figsize=(12, 3.2 * rows))
    for k, i in enumerate(show):
        plt.subplot(rows, cols, k + 1)
        plt.imshow(Image.open(test_df.iloc[i]["path"]).convert("RGB"))
        plt.title(f"সঠিক: {CLASSES[y_true[i]]}\nবলেছে: {CLASSES[y_pred[i]]} ({conf[i]:.0%})", fontsize=8)
        plt.axis("off")
    plt.tight_layout(); savefig("05_wrong_predictions.png"); plt.show()

## ১১. TFLite-এ রূপান্তর (float16, ছোট size)

In [ ]:
SAVED = str(WORK / "saved_model")
if os.path.exists(SAVED):
    shutil.rmtree(SAVED)
try:
    model.export(SAVED)                       # Keras 3
except AttributeError:
    model.save(SAVED, save_format="tf")       # Keras 2

converter = tf.lite.TFLiteConverter.from_saved_model(SAVED)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.target_spec.supported_types = [tf.float16]
tflite_bytes = converter.convert()

SERVER_DIR = WORK / "potato"            # server-এ ml_backend/models/potato/ হবে
SERVER_DIR.mkdir(exist_ok=True)
TFLITE_PATH = SERVER_DIR / "model.tflite"
TFLITE_PATH.write_bytes(tflite_bytes)
print(f"potato/model.tflite: {len(tflite_bytes)/1e6:.2f} MB")

In [ ]:
# TFLite model Keras-এর মতোই ঠিক চলছে কি না যাচাই (server-এ যা চলবে সেটাই)
interp = tf.lite.Interpreter(model_path=str(TFLITE_PATH))
interp.allocate_tensors()
inp_d = interp.get_input_details()[0]
out_d = interp.get_output_details()[0]
print("Input :", inp_d["shape"], inp_d["dtype"])
print("Output:", out_d["shape"], out_d["dtype"])

correct = 0
for p, y in zip(test_df["path"], test_df["y"]):
    img, _ = load_image(p, 0)
    interp.set_tensor(inp_d["index"], img.numpy()[None, ...].astype(np.float32))
    interp.invoke()
    pred = int(np.argmax(interp.get_tensor(out_d["index"])[0]))
    correct += int(pred == y)
TFLITE_ACC = correct / len(test_df)
print(f"TFLite test accuracy: {correct/len(test_df)*100:.2f}%  (Keras, TTA ছাড়া: {(probs_plain.argmax(1)==y_true).mean()*100:.2f}%)")

## ১২. Server-এর জন্য file তৈরি

In [ ]:
# app-এর potato_disease_info.dart-এর নামের সাথে মিল রাখা
BN = {
    "early_blight": "আগাম ধসা রোগ",
    "late_blight": "নাবী ধসা রোগ (মড়ক)",
    "healthy": "সুস্থ আলু গাছ",
    "not_leaf": "আলুর পাতা নয়",
}

(SERVER_DIR / "labels.txt").write_text("\n".join(CLASSES), encoding="utf-8")

meta = {
    "model": BEST_NAME,
    "model_version": f"potato-{BEST_NAME}",
    "input_size": IMG,
    "input_dtype": "float32",
    "input_range": "0-255 (scale ভেতরে করা আছে)",
    "color_order": "RGB",
    "classes": CLASSES,
    "classes_bn": [BN.get(c, c) for c in CLASSES],
    "confident_threshold": CONFIDENT_T,
    "min_threshold": MIN_T,
    "test_accuracy": round(float((y_pred == y_true).mean()), 4),
    "pld_test_accuracy": SOURCE_ACC.get("pld"),
}
(SERVER_DIR / "meta.json").write_text(
    json.dumps(meta, ensure_ascii=False, indent=2), encoding="utf-8")

# খুললেই potato/ folder — সরাসরি ml_backend/models/-এ রাখুন
with zipfile.ZipFile(WORK / "potato_server_bundle.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for f in ["model.tflite", "labels.txt", "meta.json"]:
        z.write(SERVER_DIR / f, f"potato/{f}")

# ---------- সব ফলাফল এক zip-এ ----------
metrics = {
    "best_model": BEST_NAME,
    "tta": bool(USE_TTA_FINAL),
    "keras_test_accuracy": round(TEST_ACC, 4),
    "test_accuracy_by_source": SOURCE_ACC,
    "comparison": comparison.to_dict(orient="records"),
    "tflite_test_accuracy": round(float(TFLITE_ACC), 4),
    "hard_conditions_test_accuracy": round(HARD_ACC, 4),
    "n_train": len(train_df), "n_val": len(val_df), "n_test": len(test_df),
    "classes": CLASSES,
}
(OUT / "metrics.json").write_text(json.dumps(metrics, ensure_ascii=False, indent=2), encoding="utf-8")
(OUT / "config.json").write_text(json.dumps({
    "IMG": IMG, "BATCH": BATCH, "EPOCHS_HEAD": EPOCHS_HEAD, "EPOCHS_FT": EPOCHS_FT,
    "CANDIDATES": CANDIDATES, "LABEL_SMOOTHING": LABEL_SMOOTHING, "MIXUP_ALPHA": MIXUP_ALPHA,
    "SPECS": {k: {"unfreeze": v["unfreeze"], "lr_ft": v["lr_ft"]} for k, v in SPECS.items()}, "DROP_CLASSES": DROP_CLASSES, "SEED": SEED,
    "NOT_LEAF_DIR": NOT_LEAF_DIR, "OTHER_CROPS_AS_NOT_LEAF": OTHER_CROPS_AS_NOT_LEAF,
    "OTHER_CROP_PER_FOLDER": OTHER_CROP_PER_FOLDER, "PLD_ROOT": PLD_ROOT,
    "PV_BASE": PV_BASE, "USE_PLANTVILLAGE": USE_PLANTVILLAGE,
    "CONFIDENT_T": CONFIDENT_T, "MIN_T": MIN_T,
    "RESUME": RESUME, "RESUME_MODEL": RESUME_MODEL,
}, ensure_ascii=False, indent=2), encoding="utf-8")
shutil.copytree(SERVER_DIR, OUT / "potato", dirs_exist_ok=True)
if os.path.exists(BEST):
    shutil.copy(BEST, OUT / "potato_best.keras")

ALL_ZIP = WORK / "potato_all_outputs.zip"
with zipfile.ZipFile(ALL_ZIP, "w", zipfile.ZIP_DEFLATED) as z:
    for p in sorted(OUT.rglob("*")):
        if p.is_file():
            z.write(p, p.relative_to(OUT))
print("potato_all_outputs.zip এর ভেতরে:")
for p in sorted(OUT.rglob("*")):
    if p.is_file():
        print("  ", p.relative_to(OUT), round(p.stat().st_size / 1e6, 3), "MB")
print()

print(open(SERVER_DIR / "labels.txt", encoding="utf-8").read())
print()
for f in ["potato/model.tflite", "potato/labels.txt", "potato/meta.json",
          "potato_server_bundle.zip", "potato_all_outputs.zip"]:
    print(f, round((WORK / f).stat().st_size / 1e6, 3), "MB")

## ✅ শেষ ধাপ
Kaggle-এর **Output** ট্যাব থেকে দুটো zip download করুন:
- **`potato_server_bundle.zip`** — খুলে ভেতরের `potato/` folder-টা project-এর **`ml_backend/models/`**-এ রাখুন। এতে আছে `model.tflite` (~৬ MB), `labels.txt`, `meta.json` (threshold সহ)।
- **`potato_all_outputs.zip`** — সব chart, report, confusion matrix, ভুল prediction, আর `potato_best.keras`।

তারপর server চালিয়ে পরীক্ষা করুন:
```
python -m ml_backend.api.main
curl -F "crop=potato" -F "file=@leaf.jpg" http://localhost:8000/predict
```
`/health`-এ `"potato": true` দেখালে app-এ আলু বাছলে "সার্ভার চালু" দেখাবে।

**মনে রাখবেন**
- PlantVillage ল্যাবের ছবি, তাই ওই অংশে accuracy বেশি আসবে। **PLD test-এর সংখ্যাটা** (খেতের ছবি) বাস্তবের কাছাকাছি।
- Early blight ↔ Late blight শুরুর দিকে দেখতে কাছাকাছি; confusion matrix-এ এদের দিকে নজর দিন।
- নিজের খেতের আলু পাতার ছবি দিয়েও একবার পরীক্ষা করুন।